# Mathematical Foundations of Bayes' Theorem
## Project 5: Bayes Decision Assistant (Spam Filter and Medical Test Risk Calculator)
**Course:** MCA Semester I – Mathematical Foundations with Python  
**Team Allocation:** Roll Numbers 26 to 31  

---
### 1. Introduction and Probability Axioms
Probability theory provides a rigorous mathematical framework for quantifying uncertainty.
Let $\Omega$ be the sample space and  \subseteq \Omega$ be an event. Kolmogorov's probability axioms state:
1. **Non-negativity:** (E) \ge 0$ for every event $.
2. **Normalization:** (\Omega) = 1$.
3. **Additivity:** For any countable sequence of mutually exclusive events , E_2, \dots$:
5775P\left(igcup_{i=1}^{\infty} E_iight) = \sum_{i=1}^{\infty} P(E_i)5775

### 2. Conditional Probability and Bayes' Theorem
The conditional probability of an event $ occurring given that event $ has occurred ((B) > 0$) is defined as:
5775P(A \mid B) = rac{P(A \cap B)}{P(B)}5775

Similarly, (B \mid A) = rac{P(A \cap B)}{P(A)} \implies P(A \cap B) = P(B \mid A)P(A)$. Substituting gives **Bayes' Theorem**:
5775P(A \mid B) = rac{P(B \mid A) P(A)}{P(B)}5775

Using the **Law of Total Probability**, where $ and $
eg A$ partition the sample space:
5775P(B) = P(B \mid A)P(A) + P(B \mid 
eg A)P(
eg A)5775
Hence:
5775P(A \mid B) = rac{P(B \mid A)P(A)}{P(B \mid A)P(A) + P(B \mid 
eg A)P(
eg A)}5775

### 3. Application 1: Medical Test Risk & Base Rate Neglect
Consider a diagnostic test for a condition $:
- **Sensitivity:** (+ \mid D)$ (True Positive Rate)
- **Specificity:** (- \mid 
eg D)$ (True Negative Rate) $\implies P(+ \mid 
eg D) = 1 - 	ext{Specificity}$ (False Positive Rate)
- **Prevalence:** (D)$ (Prior Probability of Disease)

The **Positive Predictive Value (PPV)** is:
5775P(D \mid +) = rac{P(+ \mid D)P(D)}{P(+ \mid D)P(D) + P(+ \mid 
eg D)P(
eg D)}5775

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def calculate_bayes(sensitivity, specificity, prevalence):
    sens = sensitivity / 100.0
    spec = specificity / 100.0
    prev = prevalence / 100.0
    p_pos_given_d = sens
    p_pos_given_not_d = 1.0 - spec
    p_pos = (p_pos_given_d * prev) + (p_pos_given_not_d * (1.0 - prev))
    ppv = (p_pos_given_d * prev) / p_pos
    return ppv * 100.0

prevalences = np.linspace(0.1, 20.0, 100)
ppv_curve = [calculate_bayes(95.0, 90.0, p) for p in prevalences]

plt.figure(figsize=(8, 4.5))
plt.plot(prevalences, ppv_curve, color="#2b5c8f", lw=2.5, label="Sensitivity=95%, Specificity=90%")
plt.axvline(1.0, color="r", linestyle="--", label="Rare Disease (Prevalence = 1.0% -> PPV = 8.76%)")
plt.title("Impact of Disease Prevalence on Positive Predictive Value (PPV)")
plt.xlabel("Disease Prevalence (%)")
plt.ylabel("P(Disease | Positive Test) (%)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()


### 4. Application 2: Text Classification with Multinomial Naive Bayes
For document classification, each document $ consists of word tokens , w_2, \dots, w_n$.
The target class  \in \{	ext{ham}, 	ext{spam}\}$. By Bayes' rule:
5775P(c \mid w_1, \dots, w_n) = rac{P(c) P(w_1, \dots, w_n \mid c)}{P(w_1, \dots, w_n)}5775

The **Naive Bayes Assumption** posits that individual word occurrences are conditionally independent given the class:
5775P(w_1, \dots, w_n \mid c) = \prod_{i=1}^n P(w_i \mid c)5775

To prevent floating-point underflow, computations are performed in log space:
5775\log P(c \mid w_1, \dots, w_n) \propto \log P(c) + \sum_{i=1}^n \log P(w_i \mid c)5775

#### Laplace (Add-1) Smoothing
If a word was never observed in training for class $, (w_i \mid c) = 0$, which causes $\log(0)$ to fail.
We apply Laplace smoothing:
5775P(w_i \mid c) = rac{	ext{count}(w_i, c) + lpha}{\sum_{w \in V} 	ext{count}(w, c) + lpha |V|}5775
where $|V|$ is the total number of unique words in the vocabulary and $lpha = 1.0$.

In [ ]:
import sys
sys.path.append("..")
from spam_detection import NaiveBayesClassifier, train_test_split_scratch
from model_evaluation import calculate_metrics

df = pd.read_csv("../data/sms_spam_cleaned.csv")
train_df, test_df = train_test_split_scratch(df, test_ratio=0.2, random_state=42)

model = NaiveBayesClassifier(alpha=1.0)
model.fit(train_df["message"].tolist(), train_df["label"].tolist())

y_test = test_df["label"].tolist()
y_pred = model.batch_predict(test_df["message"].tolist())
metrics = calculate_metrics(y_test, y_pred, pos_label="spam")

print(f"Accuracy:  {metrics['Accuracy']*100:.2f}%")
print(f"Precision: {metrics['Precision']*100:.2f}%")
print(f"Recall:    {metrics['Recall']*100:.2f}%")
print(f"F1-Score:  {metrics['F1-Score']:.4f}")


### 5. Empirical Verification: Monte Carlo Simulation
We simulate 100,000 independent trials using NumPy's pseudo-random number generator.
By the **Strong Law of Large Numbers**, as sample size  	o \infty$, the empirical sample mean converges almost surely to the theoretical expected value:
5775\lim_{N 	o \infty} \hat{P}_N(D \mid +) = P(D \mid +)5775

In [ ]:
from monte_carlo import run_monte_carlo_simulation

sim = run_monte_carlo_simulation(sensitivity=95.0, specificity=90.0, prevalence=1.0, num_samples=100000, random_seed=42)
print(f"Theoretical Bayes PPV: {sim['theoretical_ppv_pct']:.4f}%")
print(f"Monte Carlo Sim PPV:   {sim['simulated_ppv_pct']:.4f}%")
print(f"Absolute Difference:   {sim['difference_pct']:.4f}%")
